# GNN models — can every model learn?

Checks every model in `src/gnn.py` before any experiment uses it: the three graph
networks, the graph autoencoder and the two controls (a parameter-matched MLP and the
GCN on a shuffled taxonomy).

The check is deliberately easy: train on 100 cases + 100 controls and score on *those
same* 200 samples. This is not a performance number — it is a bug check. A model with
hundreds of thousands of parameters that cannot reach a training AUC above 0.95 on 200
samples cannot learn at all (wrong wiring, a detached gradient, labels misaligned with
nodes), and every later result from it would be meaningless. Nothing is written to
`reports/`.

## Setup

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import differential as diff
from src import gnn
from src import graphs as gr

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
REL_PATH = PROCESSED / "abund_relative.parquet"
CLR_PATH = PROCESSED / "abund_clr.parquet"

with open(ROOT / "config" / "params.yaml") as f:
    params = yaml.safe_load(f)
SEED = params["seed"]
gcfg = params["graph"]
ocfg = gcfg["overfit_check"]
mcfg = gcfg["models"]

import torch
import torch_geometric
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
warnings.filterwarnings("ignore")
torch.manual_seed(SEED)

print(f"seed={SEED}  torch={torch.__version__}  torch_geometric={torch_geometric.__version__}")
print(f"overfit check: {ocfg['n_samples']} samples, {ocfg['epochs']} epochs, lr={ocfg['lr']}, "
      f"pass mark train AUC > {ocfg['min_train_auc']}")

C:\Users\Jadrovski\vnp\MicroWave\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


seed=42  torch=2.14.0+cpu  torch_geometric=2.8.1
overfit check: 200 samples, 100 epochs, lr=0.003, pass mark train AUC > 0.95


## 1 — A tiny balanced subset, as graphs

100 cases + 100 controls from the within-project cohort. Every graph is built on these
200 samples only, so edges and taxon statistics come from the subset — the same way
they will come from each training fold later. The shuffled taxonomy (the control that
asks whether the *real* tree matters) is built here too: same shape, parents permuted
within each rank.

In [2]:
taxa = [c for c in pq.ParquetFile(CLR_PATH).schema.names if c != gr.SAMPLE_COL]
taxonomy = gr.build_taxonomy_edges(pd.read_parquet(INTERIM / "taxon_table.parquet"), taxa)
tax_edges, node_index, _ = taxonomy

cohort = pd.read_parquet(PROCESSED / "cohort_within_project.parquet")
cohort["sample"] = diff.sample_keys(cohort)
half = ocfg["n_samples"] // 2
tiny = cohort.groupby("disease_label", group_keys=False).sample(half, random_state=SEED)
t_samples = tiny["sample"].tolist()
t_projects = tiny["project"].to_numpy()
t_y = (tiny["disease_label"] == "case").astype(int).to_numpy()
t_idx = np.arange(len(t_samples))

t_bip = gr.build_bipartite_edges(REL_PATH, CLR_PATH, t_samples, taxa)
t_clr, _ = diff.load_clr(CLR_PATH, t_samples)
t_clr = t_clr.loc[t_samples, taxa]
t_pres = gr.presence_from_edges(t_bip[0], len(t_samples), len(taxa))
t_stats = gr.taxon_leaf_stats(t_clr.to_numpy(), t_pres)
t_sx, sample_feature_names = gr.sample_node_features(
    pd.read_parquet(INTERIM / "sample_depth.parquet"), t_samples)
harmonized = pd.read_parquet(INTERIM / "samples_harmonized.parquet")

t_common = dict(projects=t_projects, taxonomy=taxonomy, bipartite=t_bip, sample_x=t_sx,
                sample_feature_names=sample_feature_names, taxa_cols=taxa,
                leaf_stats=t_stats, y=t_y)
t_G1 = gnn.scale_sample_features(gr.build_hetero_graph("G1", t_samples, **t_common), t_idx)
t_G2 = gnn.scale_sample_features(gr.build_hetero_graph(
    "G2", t_samples, **t_common,
    knn=gr.build_knn_edges(t_clr.to_numpy(), t_samples, t_projects, gcfg["knn_k"])), t_idx)
t_G3 = gnn.scale_sample_features(gr.build_hetero_graph(
    "G3", t_samples, **t_common,
    project_x=gr.project_node_features(harmonized, sorted(set(t_projects)))[0]), t_idx)

shuffled = gnn.shuffle_taxonomy(taxonomy, SEED)
gr.check_taxonomy(shuffled[1], shuffled[0], n_leaves=419, n_internal=247)
moved = (shuffled[0][1] != tax_edges[1]).float().mean().item()

labels_t = pd.Series(t_y, index=t_samples)
ds_real = gr.build_per_sample_dataset(t_clr, taxonomy, labels_t, presence=t_pres,
                                      log10_prevalence=t_stats[0])
ds_shuf = gr.build_per_sample_dataset(t_clr, shuffled, labels_t, presence=t_pres,
                                      log10_prevalence=t_stats[0])

print(f"{len(t_samples)} samples ({t_y.sum()} cases) from {len(set(t_projects))} projects; "
      f"{t_bip[0].shape[1]:,} abundance edges")
print(f"shuffled taxonomy: still a valid 666-node forest; {moved:.0%} of parent pointers moved")

200 samples (100 cases) from 33 projects; 13,206 abundance edges
shuffled taxonomy: still a valid 666-node forest; 93% of parent pointers moved


## 2 — Every classifier memorises the subset

The three big-graph variants with GraphSAGE, G1 with GAT, the per-sample GCN on the
real and the shuffled tree, and an MLP matched in parameter count to each of the GCN
and GraphSAGE (the control that separates "the graph helped" from "a bigger model
helped").

In [3]:
def overfit(name, model, train_loader, eval_loader):
    t0 = time.time()
    torch.manual_seed(SEED)
    model, hist = gnn.fit(model, train_loader, n_out=1, epochs=ocfg["epochs"], lr=ocfg["lr"],
                          weight_decay=0.0, seed=SEED)
    prob, yy = gnn.predict(model, eval_loader)
    return dict(model=name, params=gnn.count_parameters(model),
                train_auc=roc_auc_score(yy, prob), first_loss=hist["train_loss"].iloc[0],
                last_loss=hist["train_loss"].iloc[-1], seconds=round(time.time() - t0, 1))

def graph_loader(g):
    return gnn.sample_loader(g, t_idx, batch_size=gcfg["train"]["batch_size"],
                             neighbor_sampling=gcfg["neighbor_sampling"], shuffle=True)

sage_cfg = {k: v for k, v in mcfg["graphsage"].items() if k != "aggr"}
gat_cfg = {k: v for k, v in mcfg["gat"].items() if k != "edge_dim"}
emb = gcfg["taxon_embed_dim"]
rows = []
for variant, g in (("G1", t_G1), ("G2", t_G2), ("G3", t_G3)):
    torch.manual_seed(SEED)
    rows.append(overfit(f"HeteroSAGE {variant}", gnn.HeteroSAGE(g, 1, embed_dim=emb, **sage_cfg),
                        graph_loader(g), graph_loader(g)))
torch.manual_seed(SEED)
rows.append(overfit("HeteroGAT G1", gnn.HeteroGAT(t_G1, 1, embed_dim=emb, **gat_cfg),
                    graph_loader(t_G1), graph_loader(t_G1)))

gcn_models = {}
for name, ds in (("TaxonomyGCN", ds_real), ("TaxonomyGCN shuffled tree", ds_shuf)):
    torch.manual_seed(SEED)
    gcn_models[name] = gnn.TaxonomyGCN(len(node_index), 1, embed_dim=emb, **mcfg["gcn"])
    rows.append(overfit(name, gcn_models[name], DataLoader(ds, batch_size=64, shuffle=True),
                        DataLoader(ds, batch_size=256)))

tol = mcfg["mlp_control"]["tolerance"]
for target in ("TaxonomyGCN", "HeteroSAGE G1"):
    n_target = next(r["params"] for r in rows if r["model"] == target)
    mlp = gnn.matched_mlp(len(taxa), 1, n_target, tolerance=tol)
    rows.append(overfit(f"MLP matched to {target}", mlp,
                        gnn.tensor_loader(t_clr.to_numpy(), t_y, batch_size=64, shuffle=True),
                        gnn.tensor_loader(t_clr.to_numpy(), t_y, batch_size=256, shuffle=False)))
    assert abs(rows[-1]["params"] - n_target) <= tol * n_target

overfit_df = pd.DataFrame(rows)
assert (overfit_df["train_auc"] > ocfg["min_train_auc"]).all(), overfit_df
overfit_df

,model,params,train_auc,first_loss,last_loss,seconds
0,HeteroSAGE G1,456321,1.0000,0.770593,0.003127,4.1
1,HeteroSAGE G2,522113,1.0000,0.833251,0.022389,4.5
2,HeteroSAGE G3,593921,0.9999,1.064749,0.026408,4.9
3,HeteroGAT G1,529857,1.0000,0.695396,0.105261,7.6
4,TaxonomyGCN,184706,0.9951,0.904140,0.133771,203.7
5,TaxonomyGCN shuffled tree,184706,0.9931,0.900030,0.228799,253.5
6,MLP matched to TaxonomyGCN,184921,1.0000,0.753714,0.002586,2.2
7,MLP matched to HeteroSAGE G1,456744,1.0000,0.990840,0.000023,2.7


## 3 — The graph autoencoder

The autoencoder has no labels, so its check is different: hide 15% of the abundance
edges, encode the rest, and ask it to tell the hidden real edges apart from random
absent (sample, taxon) pairs. The hidden edges are, by design, never seen as input, so
this AUC is a held-out score and the 0.95 memorisation bar does not apply. Instead it
must beat the obvious guess — "common taxa in taxon-rich samples are probably there"
(taxon prevalence × sample richness, from the visible edges) — and its loss must fall.
The source graph must come back unchanged — masking works on a copy.

In [4]:
pcfg = gcfg["pretrain"]
torch.manual_seed(SEED)
gae = gnn.GraphAutoencoder(gnn.HeteroSAGE(t_G1, 1, embed_dim=emb, **sage_cfg))
gen = torch.Generator().manual_seed(SEED)
n_edges_before = t_G1[gnn.ABUNDANCE].edge_index.shape[1]
with torch.no_grad():  # materialise the lazy input layers before building the optimiser
    gnn.gae_loss(gae, t_G1, mask_frac=pcfg["mask_frac"], neg_ratio=pcfg["neg_sampling_ratio"], generator=gen)
opt = torch.optim.Adam(gae.parameters(), lr=ocfg["lr"])
t0, losses = time.time(), []
for epoch in range(ocfg["epochs"]):
    gae.train()
    opt.zero_grad()
    loss, logit, lab = gnn.gae_loss(gae, t_G1, mask_frac=pcfg["mask_frac"],
                                    neg_ratio=pcfg["neg_sampling_ratio"], generator=gen)
    loss.backward()
    opt.step()
    losses.append(float(loss))

# score one fresh mask, against the prevalence x richness guess on the same pairs
gae.eval()
eval_gen = torch.Generator().manual_seed(SEED + 1)
visible, pos, _ = gnn.mask_abundance_edges(t_G1, pcfg["mask_frac"], eval_gen)
neg = gnn.sample_negative_pairs(t_G1[gnn.ABUNDANCE].edge_index, len(t_samples), len(taxa),
                                pos.shape[1], eval_gen)
pairs = torch.cat([pos, neg], dim=1)
truth = np.r_[np.ones(pos.shape[1]), np.zeros(neg.shape[1])]
with torch.no_grad():
    logit, _ = gae(visible, pairs)
vis = visible[gnn.ABUNDANCE].edge_index
prevalence = np.bincount(vis[1].numpy(), minlength=len(taxa)) / len(t_samples)
richness = np.bincount(vis[0].numpy(), minlength=len(t_samples))
gae_auc = roc_auc_score(truth, logit.numpy())
naive_auc = roc_auc_score(truth, prevalence[pairs[1]] * richness[pairs[0]])
print(f"GAE: loss {losses[0]:.3f} -> {losses[-1]:.3f}  [{time.time() - t0:.0f}s]")
print(f"held-out edge AUC: autoencoder {gae_auc:.3f}  vs  prevalence x richness {naive_auc:.3f}")
assert t_G1[gnn.ABUNDANCE].edge_index.shape[1] == n_edges_before
assert losses[-1] < losses[0] and gae_auc > naive_auc

GAE: loss 14.435 -> 2.545  [7s]
held-out edge AUC: autoencoder 0.908  vs  prevalence x richness 0.892


## 4 — Inductive training graph

Under leave-one-project-out, the held-out samples must not be in the graph the model
trains on. `restrict_to_samples` keeps a subset of sample nodes plus every taxon and
taxonomy node, and drops every edge touching a removed sample.

In [5]:
keep = np.arange(50)
sub = gnn.restrict_to_samples(t_G2, keep)
assert sub["sample"].num_nodes == 50 and sub["taxon"].num_nodes == 419
assert sub[gnn.ABUNDANCE].edge_index[0].max() < 50
assert sub["sample", "similar_to", "sample"].edge_index.max() < 50
print(f"restricted G2: {sub['sample'].num_nodes} samples, "
      f"{sub[gnn.ABUNDANCE].edge_index.shape[1]:,} abundance edges, "
      f"{sub['sample', 'similar_to', 'sample'].edge_index.shape[1]} k-NN edges")

restricted G2: 50 samples, 3,146 abundance edges, 170 k-NN edges


## Summary

Every model can learn: each classifier memorises the 200-sample subset, the shuffled
tree is a valid forest of the same shape, the matched MLPs sit within the parameter
tolerance, the autoencoder beats the obvious guess on edges it never saw, and the
training-only graph holds no held-out sample. The experiments use these models.